In [2]:
import os
from local.utils import *



def get_anno_datas(anno_dir, wav_scp):
    
    json_files = get_dir_files(anno_dir, '.json')



    anno_datas = []
    for json_file in json_files:
        utt = os.path.basename(json_file).replace('.json', '').replace('_mp3', '')
        if utt not in wav_scp:
            print(f'waring {utt} not in wav_scp')
            continue
        
        name = utt.replace('@10.190.101.161_0', '_ch0')
        name = name.replace('@10.190.101.161_1', '_ch1')
        anno_data = read_json_data(json_file)
        anno_data['utt'] = name
        anno_data['wav'] = wav_scp[utt]
        anno_datas.append(anno_data)

    print(f'get {len(anno_datas)} availabel data from {len(json_files)} anno files in {anno_dir}')
    
    return anno_datas



    
    

# wav_dir = '/data/nas/dataset/asr/kefu/huaian/wavs'
# wav_scp = get_wav_scp(wav_dir, '.wav')

wav_scp_file = '/data/nas/dataset/asr/kefu/huaian/wav.scp'
all_utts, wav_scp = read_wav_scp(wav_scp_file)

print(f'get {len(wav_scp.keys())} wav files')




get 163884 wav files


In [3]:

def is_avaiable_text(txt):
    
    if len(txt.strip()) == 0:
        return False
    
    if '-' in txt:
        return False
    else:
        return True


def filter_anno(anno_datas):
    avaiable_train_set = []
    avaiable_test_set = []
    for anno in anno_datas:
        utt = anno['utt']
        segs_anno = anno['segs']
        flag_avaiable_test = True
        for seg_anno in segs_anno:
            st,ed = seg_anno['seg']
            txt = seg_anno['text']
            
            if len(txt.strip()) == 0:
                flag_avaiable_test = False
                break

            if '-' in txt:
                # print(f'utt {utt} seg {st} {ed} {txt}')
                flag_avaiable_test = False
                break
        if flag_avaiable_test:
            avaiable_test_set.append(anno)
        else:
            avaiable_train_set.append(anno)
    
    print(f'total {len(anno_datas)},  {len(avaiable_train_set)} for train , {len(avaiable_test_set)} for test')
    return avaiable_train_set, avaiable_test_set




# 随机划分训练集，开发集， 测试集
(仅一次使用，后续测试集不变)

In [76]:
import random

anno_checked_dir = '/data/nas/dataset/asr/kefu/huaian3/annos/anno_checked'

checked_anno_datas = get_anno_datas(anno_checked_dir, wav_scp)
avaiable_train_set, avaiable_test_set = filter_anno(checked_anno_datas)

dataset_ratio = [0.6, 0.1, 0.3]

num_train = int(len(checked_anno_datas) * dataset_ratio[0])
num_test = int(len(checked_anno_datas) * dataset_ratio[2])
test_annos = avaiable_test_set[:num_test]

other_annos = avaiable_train_set + avaiable_test_set[num_test:]
random.shuffle(other_annos)

train_annos = other_annos[:num_train]
dev_annos = other_annos[num_train:]



print(f'total {len(checked_anno_datas)} , train {len(train_annos)} , dev {len(dev_annos)} , test {len(test_annos)}')


get 2000 availabel data from 2000 anno files in /data/nas/dataset/asr/kefu/huaian3/annos/anno_checked
total 2000,  464 for train , 1536 for test
total 2000 , train 1200 , dev 200 , test 600


## 后续划分 训练 开发 测试集


In [4]:
import random 



anno_checked_dir = '/data/nas/dataset/asr/kefu/huaian/annos/anno100h/'
utt_list_file = f'/data/nas/dataset/asr/kefu/huaian/utt.test'

utt_xf_file = '/data/nas/dataset/asr/kefu/huaian/utt.test.xf'

train_utt_list = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/train/utt.list'
dev_utt_list = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/dev/utt.list'
test_utt_list = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/test/utt.list'

# 读取测试集
test_utts = read_utts(utt_list_file)
test_utts2 = read_utts(utt_xf_file)


checked_anno_datas = get_anno_datas(anno_checked_dir, wav_scp)
avaiable_train_set, avaiable_test_set = filter_anno(checked_anno_datas)

avaiable_test_utts = set([data['utt'] for data in avaiable_test_set])

for test_utt in test_utts:
    if test_utt not in avaiable_test_utts:
        print(f'warning: {test_utt} not in avaiable_test_utts')
        
for test_utt in test_utts2:
    if test_utt not in avaiable_test_utts:
        print(f'warning2: {test_utt} not in avaiable_test_utts')

# 将两个测试集合并一起
test_utts = test_utts + test_utts2

test_annos = []
other_annos = avaiable_train_set
with open(f'{utt_list_file}.filter', 'w') as f:
    ori_test_utts = set(test_utts)
    for anno_data in avaiable_test_set:
        utt = anno_data['utt']
        if utt in ori_test_utts:
            f.write(f'{utt}\n')
            test_annos.append(anno_data)
        else:
            other_annos.append(anno_data)

print(f'test_annos: {len(test_annos)}, other_annos: {len(other_annos)}')

        
dataset_ratio = [0.95, 0.05]
random.shuffle(other_annos)
num_train = int(len(other_annos)*dataset_ratio[0])
train_annos = other_annos[:num_train]
dev_annos = other_annos[num_train:]

print(f'train {len(train_annos)} dev {len(dev_annos)} test  {len(test_annos)}')




get 10293 availabel data from 10293 anno files in /data/nas/dataset/asr/kefu/huaian/annos/anno100h/
total 10293,  2583 for train , 7710 for test
test_annos: 1022, other_annos: 9271
train 8807 dev 464 test  1022


In [4]:
# 已经完全划分好 train dev test 的数据集

anno_checked_dir = '/data/nas/dataset/asr/kefu/huaian/annos/anno100h/'

train_utt_list = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/train/utt.list'
dev_utt_list = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/dev/utt.list'
test_utt_list = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/test/utt.list'

train_utts = read_utts(train_utt_list)
dev_utts = read_utts(dev_utt_list)
test_utts = read_utts(test_utt_list)


# 测试集核验
checked_anno_datas = get_anno_datas(anno_checked_dir, wav_scp)
avaiable_train_set, avaiable_test_set = filter_anno(checked_anno_datas)
avaiable_test_utts = set([data['utt'] for data in avaiable_test_set])
for test_utt in test_utts:
    if test_utt not in avaiable_test_utts:
        print(f'warning: {test_utt} not in avaiable_test_utts')
        
utt2anno = {}
for anno in checked_anno_datas:
    utt = anno['utt']
    if utt in utt2anno:
        print(f'warning: {utt} has more than one anno')
    utt2anno[utt] = anno




get 10293 availabel data from 10293 anno files in /data/nas/dataset/asr/kefu/huaian/annos/anno100h/
total 10293,  2583 for train , 7710 for test


# 将数据写到文件

In [7]:
test_set1 = []
test_set2 = []

# 读取测试集
test_utts = read_utts(utt_list_file)
test_utts2 = read_utts(utt_xf_file)
set1 = set(test_utts)
set2 = set(test_utts2)

for anno_data in avaiable_test_set:
    utt = anno_data['utt']
    if utt in set1:
        test_set1.append(anno_data)
    elif utt in set2:
        test_set2.append(anno_data)

print(len(test_set1), len(test_set2))


with open(f'{anno_checked_dir}/all_anno.jsonl', 'w') as f:
    for data in checked_anno_datas:
        f.write(json.dumps(data, ensure_ascii=False) + '\n')


595 427


In [46]:
import os
import soundfile as sf
import tqdm


def get_dur_info(anno):
    utt = anno['utt']
    wav_path = anno['wav']
    if not os.path.exists(wav_path):
        return 0, 0
    speech, fs = sf.read(wav_path)
    speech_dur = len(speech)/fs
    
    no_sil_dur = 0
    for i, seg in enumerate(anno['segs']):
        st, ed = seg['seg']
        no_sil_dur += (ed-st)/1000
    
    return speech_dur, no_sil_dur
    

def split_audio_by_timestamps(anno, out_dir):
    """依据标注的时间戳，对音频进行切分"""
    utt = anno['utt']
    wav_path = anno['wav']
    if not os.path.exists(wav_path):
        return
    speech, fs = sf.read(wav_path)
    num_sample_ms = int(fs/1000)
    
    res = []
    

    for i, seg in enumerate(anno['segs']):
        st, ed = seg['seg']
        txt = remove_punctuation(seg['text'])
        
        if not is_avaiable_text(txt):
            continue
        
        # print(f'[{st}, {ed}]: {txt}')
        seg_speech = speech[int(st*num_sample_ms):int(ed*num_sample_ms)]
        seg_utt = f'{out_dir}/{utt}_seg{i:03d}'
        sf.write(f'{seg_utt}.wav', seg_speech, fs)
        with open(f'{seg_utt}.txt', 'w') as f:
            f.write(f'{txt}\n')

        
        res.append({"utt": f'{utt}_seg{i:03d}', 
                    "txt": txt, 
                    "dur": ed-st, 
                    'wav': f'{seg_utt}.wav'})
    

    
    return res, len(speech)/fs


def split_audio_by_timestamps_batch(annos, out_seg_wav_dir):
    total_seg_dur = 0
    total_speech_dur = 0
    datas = []
    for wav_anno in tqdm.tqdm(annos):
        res, speech_dur = split_audio_by_timestamps(wav_anno, out_seg_wav_dir)
        datas.extend(res)
        seg_dur = sum([x['dur'] for x in res])/1000
        total_seg_dur += seg_dur
        total_speech_dur += speech_dur

    # print(f'total seg dur: {total_seg_dur} speech_dur {total_speech_dur}')
    
    return total_seg_dur, total_speech_dur, datas

def write_data(datas, out_dir):
    
    with open(f'{out_dir}/text', 'w') as f_text, open(f'{out_dir}/wav.scp', 'w') as f_wav:
        for data in datas:
            utt = data['utt']
            txt = data['txt']
            wav = data['wav']
            
            f_text.write(f'{utt} {txt}\n')
            f_wav.write(f'{utt} {wav}\n')


In [47]:
import os


out_seg_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data/'
out_seg_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data2/'
out_seg_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian/data/'
out_seg_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian/data100h/'     # 最终标注结果

names = ['train', 'dev', 'test']
data_annos = [train_annos, dev_annos, test_annos]

for name, annos in zip(names, data_annos):
    sub_dir = f'{out_seg_wav_dir}/{name}/wavs/'
    if not os.path.exists(sub_dir):
        os.makedirs(sub_dir, exist_ok=True)
        
    total_seg_dur, total_speech_dur, seg_datas = split_audio_by_timestamps_batch(annos, sub_dir)
    write_data(seg_datas, f'{out_seg_wav_dir}/{name}/')
    ratio = total_seg_dur / total_speech_dur
    print(f'{name} total {len(seg_datas)} seg_wav, seg_dur: {total_seg_dur/3600} speech_dur: {total_speech_dur/3600} ratio: {ratio}')

100%|██████████| 8807/8807 [50:33<00:00,  2.90it/s]  


train total 177426 seg_wav, seg_dur: 87.3611775 speech_dur: 264.84617777777845 ratio: 0.32985628953762425


100%|██████████| 464/464 [02:42<00:00,  2.85it/s]


dev total 9402 seg_wav, seg_dur: 4.768080833333333 speech_dur: 14.221288888888898 ratio: 0.3352776861919061


100%|██████████| 1022/1022 [04:32<00:00,  3.76it/s]

test total 20014 seg_wav, seg_dur: 9.522925833333334 speech_dur: 29.553200000000015 ratio: 0.32222993900265723


## 过滤训练数据中较长的数据

In [50]:
from collections import defaultdict

# 读取处理好的audio_data.jsonl文件， 过滤掉长音频
# {"key": "feat5_kefu_biaobei_data01_filter-1ed28d89548f64bc695bf17b456ad402_ch0_seg013", "source": "/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data2//train/wavs//1ed28d89548f64bc695bf17b456ad402_ch0_seg013.wav", "source_len": 208, "target": "真 的 很 抱 歉 请 问 您 还 有 什 么 需 要 咨 询 的 吗", "target_len": 18}
audio_jsonl = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/exp6_1/data/train/audio_datasets.jsonl'
audio_jsonl = '/data/nas/zhangjiayuan/experiment/paraformer_finitune/exp8/data/dev/audio_datasets.jsonl'

out_jsonl = audio_jsonl.replace('.jsonl', '_filter.jsonl')

ms_datas = []
ms_datas_filter = []
count = defaultdict(int)

with open(audio_jsonl, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        ms_datas.append(data)
        dur = data['source_len']
        if dur < 8000:
            ms_datas_filter.append(data)
        dur = int(dur/1000.0)
        count[dur] += 1
        
print(count)


with open(out_jsonl, 'w') as f:
    for data in ms_datas_filter:
        f.write(json.dumps(data, ensure_ascii=False) + '\n')


defaultdict(<class 'int'>, {0: 13311, 1: 72, 2: 9, 3: 4, 9: 1, 4: 5})


In [30]:
print(test_annos[0]['segs'])
print(test_annos[0])

[{'seg': [11482, 21490], 'text': '啊，你好，打扰了啊，麻烦帮我核实一下，一个快件的取件码，它显示三十一号就签收了，但是客户说没有收到，核实一下取件码，后五位是八二六零八，圆通客服啊。'}, {'seg': [23312, 23679], 'text': '对。'}, {'seg': [24350, 26300], 'text': '对，后位八二六零八。'}, {'seg': [43794, 47052], 'text': '那你，那你跟客户解释一下呗，我把他的电话给你好吧？'}, {'seg': [49286, 49749], 'text': '哦。'}, {'seg': [52541, 58865], 'text': '但因为他没有收到，就是比较生气在这边，我看他脾气还蛮激动的，所以我立马联系了，麻烦您现在这边给他联系一下啊。'}, {'seg': [61096, 66042], 'text': '啊，我，我也要给他回复一下，然后这个件就是说，是明天能给他送过去，是吗？'}, {'seg': [69793, 70164], 'text': '这跟。'}, {'seg': [70314, 70632], 'text': '啊。'}, {'seg': [74536, 74933], 'text': '哦。'}, {'seg': [76593, 77607], 'text': '哦，好好好。'}, {'seg': [78221, 78675], 'text': '就是说。'}, {'seg': [83369, 86414], 'text': '好，我已经跟他解释了，就明天，明天上午是吧？可以吗？'}, {'seg': [87733, 90089], 'text': '嗯，好好好，行行行，谢谢啊，我给他联系一下啊。'}, {'seg': [91056, 91781], 'text': '嗯，再见。'}]
{'segs': [{'seg': [11482, 21490], 'text': '啊，你好，打扰了啊，麻烦帮我核实一下，一个快件的取件码，它显示三十一号就签收了，但是客户说没有收到，核实一下取件码，后五位是八二六零八，圆通客服啊。'}, {'seg': [23312, 23679], 'text': '对。'}, {'seg': [2

In [48]:
# 生成长音频的 text wav.scp  用于语音识别服务测试


test_long_wav_dir = f'{out_seg_wav_dir}/test_long/'

if not os.path.exists(test_long_wav_dir):
        os.makedirs(test_long_wav_dir, exist_ok=True)


long_wav_datas = []
for anno in test_annos:
    wav_file = anno['wav']
    utt = anno['utt']
    # full_text = anno['full_text']
    full_text = ""
    for seg in sorted(anno['segs'], key=lambda x: x['seg'][0]):
        
        full_text += seg['text']
    
    # print(f'{utt} {wav_file} {full_text}')
    long_wav_datas.append([utt, wav_file, full_text])

    
with open(f'{test_long_wav_dir}/wav.scp', 'w') as f_wav, open(f'{test_long_wav_dir}/text', 'w') as f_text:
    for utt, wav_file, full_text in long_wav_datas:
        f_wav.write(f'{utt} {wav_file}\n')
        f_text.write(f'{utt}\t{full_text}\n')

验证数据集中是否重叠

In [58]:

data_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data'

train_dir = f'{data_dir}/train'
dev_dir = f'{data_dir}/dev'
test_dir = f'{data_dir}/test'


def read_wav_scp(scp_file):
    utts = []
    wav2scp = {}
    
    with open(scp_file, 'r', encoding='utf-8') as f:
        for line in f:
            utt, wav_path = line.strip().split()
            utts.append(utt)
            wav2scp[utt] = wav_path
    
    return utts, wav2scp

train_utts, train_wav2scp = read_wav_scp(f'{train_dir}/wav.scp')
dev_utts, dev_wav2scp = read_wav_scp(f'{dev_dir}/wav.scp')
test_utts, test_wav2scp = read_wav_scp(f'{test_dir}/wav.scp')

if len(set(train_utts) & set(dev_utts)):
    overlab_utts = list(set(train_utts) & set(dev_utts))
    print(f'train_utts & dev_utts overlap: {len(overlab_utts)} :{overlab_utts[:3]}')

if len(set(train_utts) & set(test_utts)):
    overlab_utts = list(set(train_utts) & set(test_utts))
    print(f'train_utts & test_utts overlap: {len(overlab_utts)} :{overlab_utts[:3]}')

train_par_utts = [x.split('_seg')[0] for x in train_utts]
dev_par_utts = [x.split('_seg')[0] for x in dev_utts]
test_par_utts = [x.split('_seg')[0] for x in test_utts]

if len(set(train_par_utts) & set(dev_par_utts)):
    overlab_utts = list(set(train_par_utts) & set(dev_par_utts))
    print(f'train_par_utts & dev_par_utts overlap: {len(overlab_utts)} :{overlab_utts[:3]}')
    
if len(set(train_par_utts) & set(test_par_utts)):
    overlab_utts = list(set(train_par_utts) & set(test_par_utts))
    print(f'train_par_utts & test_par_utts overlap: {len(overlab_utts)} :{overlab_utts[:3]}')


In [57]:
print(train_utts[:3])

['3dea46826b85abd9722fce121b84fa5f_ch0_seg000', '3dea46826b85abd9722fce121b84fa5f_ch0_seg001', '3dea46826b85abd9722fce121b84fa5f_ch0_seg002']


# 提取客服的首尾句标注结果

In [ ]:
import soundfile as sf

anno_checked_dir = '/data/nas/dataset/asr/kefu/huaian3/annos/anno_checked'

all_anno_datas = get_anno_datas(anno_checked_dir, wav_scp)

utt2anno = {}
for anno_data in all_anno_datas:
    utt = anno_data['utt']
    utt2anno[utt] = anno_data
    
    
test_long_wav_dir = '/data/nas/dataset/asr-train/kefu_train/huaian_biaobei_data01/data2/test_long/'

long_utts, long_wav2scp = read_wav_scp(f'{test_long_wav_dir}/wav.scp')

first_utts = []
final_utts = []

for utt in long_utts:
    if utt not in utt2anno:
        print(f'warning: {utt} not in anno_checked_dir')
        
    if '_ch0' not in utt:
        continue
    
    anno_data = utt2anno[utt]
    wav_path = anno_data['wav']
    
    
    
    if len(anno_data['segs']) < 2:
        continue
    
    
    
    first_seg = anno_data['segs'][0]
    first_utt = "{}_seg{:03d}".format(utt, 0)
    final_seg = anno_data['segs'][-1]
    final_utt = "{}_seg{:03d}".format(utt, len(anno_data['segs'])-1)

    if first_seg['seg'][0] > 10000:
        continue
    
    speech, fs = sf.read(wav_path)
    
    if final_seg['seg'][0] < int(len(speech)/fs*1000) - 10000:
        continue
    
    first_seg['seg_utt'] = first_utt
    first_seg['utt'] = utt
    final_seg['seg_utt'] = final_utt
    final_seg['utt'] = utt
    
    first_utts.append(first_utt)
    final_utts.append(final_utt)
    
    # print(anno_data)
    print(first_seg, final_seg)
    # print(final_seg)
    
    # break

print(f'total {len(first_utts)} first utts {len(final_utts)} final utts')

with open(f'{test_long_wav_dir}/first_utts.txt', 'w') as f:
    for utt in first_utts:
        f.write(utt + '\n')
    
with open(f'{test_long_wav_dir}/final_utts.txt', 'w') as f:
    for utt in final_utts:
        f.write(utt + '\n')

# 驾驶舱数据

In [26]:
import os


cockpit_wav_dir = '/data/nas/dataset/asr/cockpit/product20250508/'

wav_list = get_dir_files(cockpit_wav_dir, '.wav')

print(wav_list[:10])

cockpit_datas = []
for wav_file in wav_list:
    txt_file = wav_file[:-4] + '.txt'
    if not os.path.isfile(txt_file):
        continue
    utt = wav_file.split('/')[-1][:-4]
    with open(txt_file, 'r' ) as f:
        txt = f.read().strip()
        txt = remove_punctuation(txt)
        cockpit_datas.append({
            'utt': utt,
            'txt': txt,
            'wav': wav_file
        })

for data in cockpit_datas[:4]:
    print(data)

['/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895106382249824256.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895510168734998528.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895516613417951232.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895521545277087744.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895541246371704832.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895553692708556800.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895553801492074496.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895560483773321216.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895560506376597504.wav', '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895567323156652032.wav']
{'utt': 'speech-1895106382249824256', 'txt': '十分工作', 'wav': '/data/nas/dataset/asr/cockpit/product20250508/wavs/speech-1895106382249824256.wav'}
{'ut

## 分析 淮安客服-包含讯飞结果 的数据

In [38]:
from openpyxl import load_workbook


excel_file = '/home/zhangjiayuan/temp/VoiceDocument0506.xlsx'
out_dir = '/data/nas/dataset/asr/kefu/huaian/'

wb = load_workbook(excel_file)
sheet = wb.active  # 获取第一个工作表

# 打印表头（首行）
header = [cell.value for cell in sheet[1]]
print("表头:", header)

# 打印总行数（排除空行）
print("总行数:", sheet.max_row)


column_names = ['sessionId', 'caller', '科讯转写', '自研转写']
column_indexs =[header.index(column_name) + 1 for column_name in column_names]  # 列号从1开始
print(column_indexs)

datas = []
# 循环打印第i行第j列（示例：打印前5行第1列）
for i in range(2, sheet.max_row + 1):  # 从第2行开始，限制5行
    data = [sheet.cell(row=i, column=j).value for j in column_indexs]
    
    utt = data[0]
    if utt[-2:] == "_0":
        utt = utt[:-2] + "_ch0"
    elif utt[-2:] == "_1":
        utt = utt[:-2] + "_ch1"
    else:
        print(f'error: utt format error: {utt}')
    hyp_kexun = data[2].replace('客服:', '').replace('客户:', '').replace('\n', '')
    hyp_ziyan = data[3].replace('客服:', '').replace('客户:', '').replace('\n', '')
    # print(f'{data[0]}\n{hyp_kexun}\n{hyp_ziyan}')
    
    datas.append([utt, hyp_kexun, hyp_ziyan])
    
    # break
    


表头: ['sessionId', 'caller', '科讯转写', '自研转写']
总行数: 19923
[1, 2, 3, 4]


In [43]:
# 读取 淮安-讯飞 标注结果

anno_checked_dir = '/data/nas/dataset/asr/kefu/huaian/annos/anno100h/'


checked_anno_datas = get_anno_datas(anno_checked_dir, wav_scp)
avaiable_train_set, avaiable_test_set = filter_anno(checked_anno_datas)
avaiable_test_utts = set([data['utt'] for data in avaiable_test_set])

avaiable_datas = []
for utt, hyp_kexun, hyp_ziyan in datas:
    if utt in avaiable_test_utts:
        avaiable_datas.append([utt, hyp_kexun, hyp_ziyan])

print(f'{len(avaiable_datas)} avaiable data, have biaobei-kexun')

avaiable_utts = set([data[0] for data in avaiable_datas])

total_speech_dur = 0
total_no_sil_dur = 0
for anno in checked_anno_datas:
    utt = anno['utt']
    if utt not in avaiable_utts:
        continue
    speech_dur, no_sil_dur = get_dur_info(anno)
    total_speech_dur += speech_dur
    total_no_sil_dur += no_sil_dur

total_speech_dur /= 3600.0
total_no_sil_dur /= 3600.0
print(f'total speech dur: {total_speech_dur}, total no sil dur: {total_no_sil_dur}')    

with open(f'{out_dir}/hyp_kexun', 'w') as f_kexun, open(f'{out_dir}/hyp_ziyan', 'w') as f_ziyan:
    for utt, hyp_kexun, hyp_ziyan in avaiable_datas:
        f_kexun.write(f'{utt}\t{hyp_kexun}\n')
        f_ziyan.write(f'{utt}\t{hyp_ziyan}\n')

get 10293 availabel data from 10293 anno files in /data/nas/dataset/asr/kefu/huaian/annos/anno100h/
total 10293,  2583 for train , 7710 for test
427 avaiable data, have biaobei-kexun
total speech dur: 12.568400000000002, total no sil dur: 4.1746933333333365


In [42]:
total_speech_dur /= 3600.0
total_no_sil_dur /= 3600.0
print(f'total speech dur: {total_speech_dur}, total no sil dur: {total_no_sil_dur}')   

total speech dur: 12.568400000000002, total no sil dur: 4.1746933333333365
